In [1]:
#import necessary package
import torch
import copy
import torch.nn.utils.prune as prune
from torchvision import transforms, datasets, models
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [4]:
# preprocess images
transform = transforms.Compose([
    transforms.Resize((224, 224)),   # MobileNetV2 default input size
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], 
                         [0.229, 0.224, 0.225])  # ImageNet normalization
])

train_dataset = datasets.ImageFolder("../data/dataset/Training", transform=transform)
test_dataset   = datasets.ImageFolder("../data/dataset/Test", transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader   = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)
# Load model
model = torch.load("fruit_mobilenetv2.pth", weights_only=False)

# Training SetUp
# criterion = nn.CrossEntropyLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)
model


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [8]:
# Parameterizable Leaky ReLU
class PLReLU(nn.Module):
    def __init__(self, init_alpha=0.1, init_bias=0.0):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(init_alpha))
        self.bias  = nn.Parameter(torch.tensor(init_bias))

    def forward(self, x):
        x_shifted = x - self.bias
        return torch.where(x_shifted > 0, x_shifted, self.alpha * x_shifted)


In [9]:
# static quantization float32 -> int4
class Static4BitQuant(nn.Module):
    def __init__(self):
        super().__init__()
        self.initialized = False
        self.register_buffer("scale", torch.tensor(1.0))

    def forward(self, x):
        # Eq. (3): compute scale once
        if self.training and not self.initialized:
            max_val = x.detach().abs().max()
            self.scale = max_val / 7.0 if max_val > 0 else torch.tensor(1.0, device=x.device)
            self.initialized = True

        # Eq. (4): quantize
        q = torch.round(x / self.scale)

        # Eq. (5): clamp to 4-bit range
        q = torch.clamp(q, -8, 7)

        # Eq. (6): dequantize
        return q * self.scale


In [10]:
# Static 4bit quantization 8 bits split to 4 bits
class BitSplit4BitQuant(nn.Module):
    def __init__(self):
        super().__init__()
        self.initialized = False
        self.register_buffer("scale", torch.tensor(1.0))

    def forward(self, x):
        # Compute scale once
        if self.training and not self.initialized:
            max_val = x.detach().abs().max()
            self.scale = max_val / 127.0   # simulate 8-bit
            self.initialized = True

        # Simulate 8-bit quantization
        q8 = torch.round(x / self.scale).clamp(-128, 127)

        # Split into two 4-bit halves
        # Lower 4 bits
        low = (q8 & 0x0F) - 8
        # Upper 4 bits
        high = ((q8 >> 4) & 0x0F) - 8

        # Stack into two channels
        q4 = torch.stack([low, high], dim=1)  # shape: [N, 2, C, H, W]

        # Dequantize
        return q4 * self.scale


In [11]:
# def apply_paper5_quantization(model):
#     for name, module in model.features.named_children():
#         if hasattr(module, "conv"):
#             new_layers = []
#             for sub in module.conv:
#                 if isinstance(sub, nn.ReLU6) or isinstance(sub, nn.ReLU):
#                     new_layers.append(PLReLU())
#                     new_layers.append(Static4BitQuant())  # or BitSplit4BitQuant()
#                 else:
#                     new_layers.append(sub)
#             module.conv = nn.Sequential(*new_layers)
#     return model

def replace_relu_with_paper5(module):
    for name, child in module.named_children():

        # If this child IS a ReLU or ReLU6 → replace it
        if isinstance(child, nn.ReLU6) or isinstance(child, nn.ReLU):
            setattr(module, name, nn.Sequential(
                PLReLU(),
                Static4BitQuant()
            ))

        else:
            # Otherwise, go deeper into the module
            replace_relu_with_paper5(child)


def apply_paper5_quantization(model):
    replace_relu_with_paper5(model)
    return model



In [ ]:
# Apply quantization to model
model = apply_paper5_quantization(model)
model.to(device)


In [15]:
model = torch.load("backup2_fruit_mobilenetv2.pth", weights_only=False)
for m in model.modules():
    if isinstance(m, torch.nn.BatchNorm2d):
        print("mean:", m.running_mean[:5])
        print("var :", m.running_var[:5])
        break
model.to(device)

mean: tensor([-0.0225,  0.0071,  0.0446, -0.0040, -0.3770])
var : tensor([0.1519, 0.0622, 0.6975, 0.0633, 6.3670])


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): Sequential(
        (0): PLReLU()
        (1): Static4BitQuant()
      )
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): Sequential(
            (0): PLReLU()
            (1): Static4BitQuant()
          )
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActiva

In [16]:

# evaluate function
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total
# Quantization Aware Training -> train model to adapt with the noise of 4 bit quantization
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3, momentum=0.9)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    acc = evaluate(model, test_loader, device)
    print(f"Epoch {epoch+1}, Test Accuracy: {acc:.2f}%")


Epoch 1, Test Accuracy: 38.18%
Epoch 2, Test Accuracy: 41.12%
Epoch 3, Test Accuracy: 41.82%
Epoch 4, Test Accuracy: 41.99%


KeyboardInterrupt: 

In [ ]:
# Export to ONNX
dummy = torch.randn(1, 3, 224, 224).to(device)
torch.onnx.export(model, dummy, "../model/quantization_mobilenetv2_4bit.onnx",
                  input_names=["input"], output_names=["output"], opset_version=13)
